# 迷你股价数据库构建

基于akshare库构建数据

In [ ]:
import akshare as ak   # 通过pip install akshare安装

years = [2024, 2025]  # 要获取数据的年份
stock_list = [('601398','工商银行'), ('600519','贵州茅台'), ('300750','宁德时代')] # 要获取数据的(股票代码、公司名称）二元组列表
stock_data_path = 'stock_data.txt' # 存储地址
with open(stock_data_path, 'w') as fw:
    for stock_id, company in stock_list:
        for year in years:
            df = ak.stock_zh_a_hist(symbol=stock_id, period='monthly',
                start_date=f'{year}0101', end_date=f'{year}1231', adjust='qfq')
            texts = [f"日期{row['日期']}，收盘价{row['收盘']}元。" for _,row in df.iterrows()]
            texts.insert(0, f'{year}年{company}（{stock_id}）月度股票行情数据。')
            fw.write(''.join(texts)  + '\n')


基于腾讯财经爬取数据

In [9]:
import requests   

years = [2024, 2025]  # 要获取数据的年份
stock_list = [('601398','工商银行'), ('600519','贵州茅台'), ('300750','宁德时代')] # 要获取数据的(股票代码、公司名称）二元组列表
stock_data_path = 'stock_data_tencent.txt' # 存储地址

def get_market_prefix(stock_id):
    """由6位股票代码推断交易所前缀：sh（上交所）/sz（深交所）/bj（北交所）"""
    if stock_id.startswith(('60', '68', '90', '11', '13')):
        return 'sh'
    if stock_id.startswith(('00', '30', '20', '12')):
        return 'sz'
    return 'bj'

def get_monthly_kline(stock_id, start_date, end_date, adjust='qfq'):
    symbol = get_market_prefix(stock_id) + stock_id
    if adjust:
        url = 'https://web.ifzq.gtimg.cn/appstock/app/fqkline/get'
        param = f'{symbol},month,{start_date},{end_date},1000,{adjust}'
        key = f'{adjust}month'
    else:
        url = 'https://web.ifzq.gtimg.cn/appstock/app/kline/kline'
        param = f'{symbol},month,{start_date},{end_date},1000'
        key = 'month'
    data = requests.get(url, params={'param': param}, timeout=10).json()
    return data['data'][symbol].get(key, [])

with open(stock_data_path, 'w') as fw:
    for stock_id, company in stock_list:
        for year in years:
            klines = get_monthly_kline(stock_id, f'{year}-01-01', f'{year}-12-31', adjust='qfq')
            texts = [f"日期{kline[0]}，收盘价{float(kline[2])}元。" for kline in klines]
            texts.insert(0, f'{year}年{company}（{stock_id}）月度股票行情数据。')
            fw.write(''.join(texts)  + '\n')


# 向量化公司股价知识库

In [13]:
import numpy as np
from sentence_transformers import SentenceTransformer

class TextSearchEngine():
    def __init__(self, text_encoder_path="/path/to/all-MiniLM-L6-v2"):
        self.encoder = SentenceTransformer(text_encoder_path)
        self.embeddings = np.array([]) 
        self.texts = []

    def _txt2vec(self, texts):
        return self.encoder.encode(texts)
        
    def index(self, texts): # 新增文本向量表示
        keys = [text.split('。')[0] for text in texts] # 将每条记录的第一句话向量化 
        new_embeddings = self._txt2vec(keys)
        if len(self.embeddings) == 0:
            self.embeddings = new_embeddings
        else:
            self.embeddings = np.vstack((self.embeddings, new_embeddings))
        self.texts.extend(texts)
        
    def search(self, query, top_k=2): # 查找相似文本
        query_embedding = self._txt2vec([query])
        # 此处模型已经保证embedding范数为1
        scores = np.dot(self.embeddings, query_embedding.T).flatten()
        top_indices = np.argsort(scores)[-top_k:][::-1]
        hits = [self.texts[i] for i in top_indices]
        return hits
    
texts = open(stock_data_path).readlines()
texts = [line.strip('\r\n') for line in texts]
searcher = TextSearchEngine()
searcher.index(texts)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

# 相关文档检索

In [14]:
query = "分析2025年工商银行的股价走势和特点"
hits = searcher.search(query=query, top_k=2)
for text in hits:
    print(text)

2025年工商银行（601398）月度股票行情数据。日期2025-01-27，收盘价6.345元。日期2025-02-28，收盘价6.395元。日期2025-03-31，收盘价6.415元。日期2025-04-30，收盘价6.535元。日期2025-05-30，收盘价6.595元。日期2025-06-30，收盘价7.115元。日期2025-07-31，收盘价7.25元。日期2025-08-29，收盘价7.12元。日期2025-09-30，收盘价6.99元。日期2025-10-31，收盘价7.47元。日期2025-11-28，收盘价7.8元。日期2025-12-31，收盘价7.761元。
2024年工商银行（601398）月度股票行情数据。日期2024-01-31，收盘价4.245元。日期2024-02-29，收盘价4.405元。日期2024-03-29，收盘价4.355元。日期2024-04-30，收盘价4.505元。日期2024-05-31，收盘价4.505元。日期2024-06-28，收盘价4.775元。日期2024-07-31，收盘价5.232元。日期2024-08-30，收盘价5.362元。日期2024-09-30，收盘价5.562元。日期2024-10-31，收盘价5.422元。日期2024-11-29，收盘价5.532元。日期2024-12-31，收盘价6.302元。


# 大模型基于相关文档进行回答

In [16]:
from openai import OpenAI

class LLMClient():
    def __init__(self, my_key):
        workspace_id = 'your_workspace_id'
        self.client = OpenAI(api_key=my_key, base_url=f'https://{workspace_id}.cn-beijing.maas.aliyuncs.com/compatible-mode/v1')
        
    def _process (self, prompt, model="qwen3-max"):
        messages = [{"role": "system", "content": "You are a helpful assistant."}, {"role": "user", "content":prompt}]
        completion = self.client.chat.completions.create(model=model, messages=messages)
        resp = completion.to_dict()['choices'][0]['message']['content']
        return resp

    def answer(self, prompt):
        return self._process(prompt)

from transformers import AutoModelForCausalLM, AutoTokenizer  
        
class LocalLLMClient(LLMClient):
    def __init__(self, model_path):
        self.tokenizer = AutoTokenizer.from_pretrained(model_path)
        self.model = AutoModelForCausalLM.from_pretrained(model_path, dtype="auto", device_map="auto")

    def _process(self, prompt):
        messages = [{"role": "user", "content": prompt}]
        text = self.tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
        model_inputs = self.tokenizer([text], return_tensors="pt").to(self.model.device)
        generated_ids = self.model.generate(**model_inputs, max_new_tokens=10000)
        output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
        resp = self.tokenizer.decode(output_ids, skip_special_tokens=True)
        return resp
model_path = 'local_llms/Qwen3-0.6B'
llm = LocalLLMClient(model_path)

Loading weights:   0%|          | 0/311 [00:00<?, ?it/s]

In [18]:
context = '\n'.join(hits)
prompt = f"请结合以下文本内容回答问题：{context}。\n问题为：{query}"
resp = llm.answer(prompt)
print(resp)

2025年工商银行（601398）的股价走势如下：

- 2025-01-27：收盘价6.345元  
- 2025-02-28：6.395元  
- 2025-03-31：6.415元  
- 2025-04-30：6.535元  
- 2025-05-30：6.595元  
- 2025-06-30：7.115元  
- 2025-07-31：7.25元  
- 2025-08-29：7.12元  
- 2025-09-30：6.99元  
- 2025-10-31：7.47元  
- 2025-11-28：7.8元  
- 2025-12-31：7.761元  

从这些数据可以看出，2025年工商银行的股价在短期内呈现上升趋势，特别是在2025年6月和7月，股价达到较高水平。然而，到12月时，股价有所回落，最终收盘价降至7.761元，显示出波动性。整体来看，2025年的股价走势呈现出上升、波动、随后回落的特征，但整体呈现积极向上的趋势。
